# 00 — Profile Token Distribution

**Mục đích:** Đo token length distribution của prompt + completion trên dữ liệu thực tế.  
Kết quả dùng để chốt `max_seq_length` và `max_new_tokens` trước khi train.

**Không cần GPU.** Chạy được trên máy local hoặc Colab free tier.

**Output:** `profile_results.json`, `token_distribution.png`

In [ ]:
# On Colab: uncomment and run this cell first
# !pip install transformers datasets --quiet

In [ ]:
import json
import math
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from datasets import load_dataset

# ---- Config ----
MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_tv_v6"
PROFILE_SAMPLE = 1000
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
print("Config OK")

In [ ]:
# Load tokenizer (downloads ~5MB config files, no model weights)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Vietnamese round-trip test
test_vi = "Điện thoại Samsung Galaxy S25 Ultra 256GB màu Đen Titanium"
encoded = tokenizer.encode(test_vi)
decoded = tokenizer.decode(encoded, skip_special_tokens=True)
print(f"Test string : {test_vi}")
print(f"Token count : {len(encoded)}")
print(f"Decoded back: {decoded}")
print(f"Round-trip  : {'OK' if decoded.strip() == test_vi.strip() else 'MISMATCH — check tokenizer'}")

In [ ]:
# Load dataset (public, no token needed)
ds = load_dataset(DATASET_NAME)
print(ds)

# Detect split names
splits = list(ds.keys())
val_key = "val" if "val" in splits else "validation"
print(f"\nSplits: {splits}")
print(f"Train: {len(ds['train']):,} | Val: {len(ds[val_key]):,} | Test: {len(ds['test']):,}")
print(f"Columns: {ds['train'].column_names}")

In [ ]:
# Sample 1000 random train items
train = ds["train"]
indices = random.sample(range(len(train)), PROFILE_SAMPLE)
sample = train.select(indices)
print(f"Sampled {len(sample)} items")

In [ ]:
# Prompt template (canonical — same as utils/prompt_builder.py)
PROMPT_TEMPLATE = """Sản phẩm này có giá bao nhiêu ?
Tiêu đề: {title}
Danh mục: {category}
Thương hiệu: {brand}
Mô tả: {description}
Thông số: {features}

Giá là: """


def build_prompt(item: dict) -> str:
    return PROMPT_TEMPLATE.format(
        title=item.get("title") or "",
        category=item.get("category") or "",
        brand=item.get("brand") or "Không rõ",
        description=item.get("description") or "Không có mô tả",
        features=item.get("features") or "Không có thông số",
    )


def build_completion(price: float) -> str:
    """Train/val completion: round(price/1000) as string."""
    return str(int(round(price / 1000)))


# Show one sample
item0 = dict(sample[0])
print("=== Sample prompt ===")
print(build_prompt(item0))
print(f"COMPLETION: {build_completion(item0['price'])}")

In [ ]:
# Tokenize all samples and collect lengths
from tqdm.auto import tqdm

prompt_lens = []
completion_lens = []
full_lens = []

for item in tqdm(sample, desc="Tokenizing"):
    item = dict(item)
    prompt = build_prompt(item)
    completion = build_completion(item["price"])
    
    # Note: tokenize prompt and full separately to get accurate lengths
    p_ids = tokenizer.encode(prompt, add_special_tokens=False)
    c_ids = tokenizer.encode(completion, add_special_tokens=False)
    f_ids = tokenizer.encode(prompt + completion, add_special_tokens=False)
    
    prompt_lens.append(len(p_ids))
    completion_lens.append(len(c_ids))
    full_lens.append(len(f_ids))

prompt_lens = np.array(prompt_lens)
completion_lens = np.array(completion_lens)
full_lens = np.array(full_lens)

print(f"Tokenized {len(prompt_lens)} samples")

In [ ]:
# Compute percentile stats
def pstats(arr):
    return {
        "p50": int(np.percentile(arr, 50)),
        "p90": int(np.percentile(arr, 90)),
        "p95": int(np.percentile(arr, 95)),
        "p99": int(np.percentile(arr, 99)),
        "max": int(arr.max()),
        "mean": round(float(arr.mean()), 1),
    }

prompt_stats = pstats(prompt_lens)
completion_stats = pstats(completion_lens)
full_stats = pstats(full_lens)

print("Prompt tokens    :", prompt_stats)
print("Completion tokens:", completion_stats)
print("Full tokens      :", full_stats)

In [ ]:
# Plot histogram
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, stats in zip(
    axes,
    [prompt_lens, completion_lens, full_lens],
    ["Prompt tokens", "Completion tokens", "Full (prompt+completion)"],
    [prompt_stats, completion_stats, full_stats],
):
    ax.hist(data, bins=30, edgecolor="black", color="steelblue", alpha=0.8)
    ax.axvline(stats["p95"], color="red", linestyle="--", label=f'p95={stats["p95"]}')
    ax.axvline(stats["p99"], color="orange", linestyle=":", label=f'p99={stats["p99"]}')
    ax.set_title(title)
    ax.set_xlabel("Token count")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.suptitle(f"Token distribution — Qwen3.5-4B tokenizer on {PROFILE_SAMPLE} train samples", y=1.02)
plt.tight_layout()
plt.savefig("token_distribution.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved token_distribution.png")

In [ ]:
# Compute recommendations
p95_full = full_stats["p95"]
p99_completion = completion_stats["p99"]

# Round p95_full up to nearest 64
recommended_seq = math.ceil(p95_full / 64) * 64
recommended_new_tokens = p99_completion + 1

print(f"p95 full tokens     : {p95_full}")
print(f"p99 completion toks : {p99_completion}")
print()
print(f"==> recommended max_seq_length  = {recommended_seq}")
print(f"==> recommended max_new_tokens  = {recommended_new_tokens}")
print()
print("English reference (items_llm_v6): prompt ~110-128 tokens, completion 1-3 tokens")
print(f"Vietnamese ratio vs English (prompt p95): {full_stats['p95']} / ~128 = {full_stats['p95']/128:.2f}x")

In [ ]:
# Save profile_results.json
results = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "sample_size": PROFILE_SAMPLE,
    "seed": SEED,
    "prompt_tokens": prompt_stats,
    "completion_tokens": completion_stats,
    "full_tokens": full_stats,
    "recommended_max_seq_length": recommended_seq,
    "recommended_max_new_tokens": recommended_new_tokens,
}

with open("profile_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved profile_results.json")
print(json.dumps(results, indent=2, ensure_ascii=False))

## Ket qua Phase 0 Notebook 1

**STOP:** Share noi dung `profile_results.json` va `token_distribution.png` voi Claude.  
Claude se confirm `max_seq_length` va `max_new_tokens` truoc khi sang Notebook 01.